In [1]:
!pip install lightgbm tensorflow scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
import kagglehub

path = kagglehub.dataset_download("aryayadav0513/m5-forecasting-accuracy")
print("Path:", path)

In [ ]:
import os

os.listdir(path)

In [ ]:
os.listdir(f"{path}/m5-forecasting-accuracy")

In [ ]:
import pandas as pd

In [ ]:
sales = pd.read_csv(f"{path}/m5-forecasting-accuracy/sales_train_validation.csv")
calendar = pd.read_csv(f"{path}/m5-forecasting-accuracy/calendar.csv")
prices = pd.read_csv(f"{path}/m5-forecasting-accuracy/sell_prices.csv")

In [ ]:
import os

dataset_path = os.path.join(path, "m5-forecasting-accuracy")

sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:
print(sales.shape)
print(calendar.shape)
print(prices.shape)

In [ ]:
print(path)
os.listdir(path)

In [ ]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:
# ============================================
# 1. FILTER DATA
# ============================================

sales = sales[
    (sales['state_id'].isin(['CA', 'TX', 'WI'])) &
    (sales['cat_id'].isin(['FOODS', 'HOBBIES', 'HOUSEHOLD']))
].copy()

print("Filtered sales shape:", sales.shape)

# Use ALL available days
day_cols = [c for c in sales.columns if c.startswith('d_')]

print("Number of days:", len(day_cols))

sales = sales[
    ['item_id', 'dept_id', 'cat_id',
     'store_id', 'state_id'] + day_cols
]

In [ ]:
# Filter dulu
sales_filtered = sales[
    sales['state_id'].isin(['CA', 'TX', 'WI']) &
    sales['cat_id'].isin(['FOODS', 'HOBBIES', 'HOUSEHOLD'])
].copy()

print(sales_filtered.shape)

In [ ]:
sales_long = sales_filtered.melt(
    id_vars=[
        'item_id',
        'dept_id',
        'cat_id',
        'store_id',
        'state_id'
    ],
    var_name='d',
    value_name='sales'
)

In [ ]:
sales_long = sales_long.merge(
    calendar[
        ['d', 'wm_yr_wk', 'month', 'wday']
    ],
    on='d',
    how='left'
)

In [ ]:
sales_long = sales_long.merge(
    prices,
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)

In [ ]:
# Missing prices → 0
sales_long['sell_price'] = sales_long['sell_price'].fillna(0)

In [ ]:
# Revenue
sales_long['revenue'] = (
    sales_long['sales'] *
    sales_long['sell_price']
)

# Weekly aggregation
df = (
    sales_long
    .groupby(
        ['cat_id', 'state_id', 'wm_yr_wk', 'month'],
        as_index=False
    )['revenue']
    .sum()
)

df = df.sort_values(
    ['cat_id', 'state_id', 'wm_yr_wk']
).reset_index(drop=True)

print(df.shape)
df.head()

In [ ]:
df = df.sort_values(
    ['cat_id', 'state_id', 'wm_yr_wk']
)

for lag in [1, 2, 3, 4, 8, 12]:
    df[f'lag_{lag}'] = (
        df.groupby(['cat_id', 'state_id'])['revenue']
          .shift(lag)
    )

df['rolling_mean_4'] = (
    df.groupby(['cat_id', 'state_id'])['revenue']
      .transform(lambda x: x.shift(1).rolling(4).mean())
)

df['rolling_mean_12'] = (
    df.groupby(['cat_id', 'state_id'])['revenue']
      .transform(lambda x: x.shift(1).rolling(12).mean())
)

df = df.dropna()

In [ ]:
# ============================================
# 3. FEATURE ENGINEERING
# ============================================

group_cols = ['cat_id', 'state_id']

for lag in [1, 2, 3, 4, 8, 12, 13, 26, 52]:
    df[f'lag_{lag}'] = (
        df.groupby(group_cols)['revenue']
        .shift(lag)
    )

# Rolling statistics
for window in [4, 8, 12]:
    df[f'rolling_mean_{window}'] = (
        df.groupby(group_cols)['revenue']
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    df[f'rolling_std_{window}'] = (
        df.groupby(group_cols)['revenue']
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

# Growth
df['growth_1'] = (
    df.groupby(group_cols)['revenue']
    .pct_change()
)

# Month cyclic features
df['month_sin'] = np.sin(
    2 * np.pi * df['month'] / 12
)

df['month_cos'] = np.cos(
    2 * np.pi * df['month'] / 12
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna().reset_index(drop=True)

print(df.shape)

In [ ]:
# ============================================
# 4. CHRONOLOGICAL TRAIN / CALIBRATION / TEST
# ============================================

df['group_pos'] = (
    df.groupby(group_cols)
      .cumcount()
)

df['group_size'] = (
    df.groupby(group_cols)['revenue']
      .transform('size')
)

df['train_end'] = (
    df['group_size'] * 0.70
).astype(int)

df['cal_end'] = (
    df['group_size'] * 0.85
).astype(int)

train_mask = (
    df['group_pos'] < df['train_end']
)

cal_mask = (
    (df['group_pos'] >= df['train_end']) &
    (df['group_pos'] < df['cal_end'])
)

test_mask = (
    df['group_pos'] >= df['cal_end']
)

print("Train:", train_mask.sum())
print("Calibration:", cal_mask.sum())
print("Test:", test_mask.sum())

In [ ]:
features = [
    'lag_1',
    'lag_2',
    'lag_3',
    'lag_4',
    'lag_8',
    'lag_12',
    'lag_13',
    'lag_26',
    'lag_52',
    'rolling_mean_4',
    'rolling_mean_8',
    'rolling_mean_12',
    'rolling_std_4',
    'rolling_std_8',
    'rolling_std_12',
    'growth_1',
    'month_sin',
    'month_cos'
]

X = df[features]
y = df['revenue']

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_cal = X.loc[cal_mask]
y_cal = y.loc[cal_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

model_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_lgb.fit(
    X_train,
    y_train
)

pred_lgb_cal = model_lgb.predict(X_cal)
pred_lgb_test = model_lgb.predict(X_test)

In [ ]:
model_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    random_state=42
)

model_lgb.fit(
    X_train,
    y_train
)

pred_lgb = model_lgb.predict(X_val)

print("LightGBM predictions:", pred_lgb.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_values = df.loc[
    train_mask,
    ['revenue']
]

scaler.fit(train_values)

df['revenue_scaled'] = scaler.transform(
    df[['revenue']]
)

In [ ]:
lookback = 12

X_lstm = []
y_lstm = []
target_indices = []

for (cat, state), group in df.groupby(
    ['cat_id', 'state_id']
):

    group = group.sort_values(
        'wm_yr_wk'
    )

    values = group['revenue_scaled'].values
    indices = group.index.values

    for i in range(lookback, len(values)):

        X_lstm.append(
            values[i-lookback:i]
        )

        y_lstm.append(
            values[i]
        )

        target_indices.append(
            indices[i]
        )

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)
target_indices = np.array(target_indices)

X_lstm = X_lstm.reshape(
    X_lstm.shape[0],
    X_lstm.shape[1],
    1
)

lstm_train_mask = train_mask.loc[
    target_indices
].values

lstm_cal_mask = cal_mask.loc[
    target_indices
].values

lstm_test_mask = test_mask.loc[
    target_indices
].values

X_lstm_train = X_lstm[lstm_train_mask]
y_lstm_train = y_lstm[lstm_train_mask]

X_lstm_cal = X_lstm[lstm_cal_mask]
y_lstm_cal = y_lstm[lstm_cal_mask]

X_lstm_test = X_lstm[lstm_test_mask]
y_lstm_test = y_lstm[lstm_test_mask]

idx_lstm_cal = target_indices[lstm_cal_mask]
idx_lstm_test = target_indices[lstm_test_mask]

print(X_lstm_train.shape)
print(X_lstm_cal.shape)
print(X_lstm_test.shape)

In [ ]:
# Determine whether each LSTM target belongs to train or validation
lstm_train_mask = train_mask.loc[target_indices].values
lstm_val_mask = val_mask.loc[target_indices].values

X_lstm_train = X_lstm[lstm_train_mask]
X_lstm_val = X_lstm[lstm_val_mask]

y_lstm_train = y_lstm[lstm_train_mask]
y_lstm_val = y_lstm[lstm_val_mask]

lstm_val_indices = target_indices[lstm_val_mask]

print("LSTM training:", X_lstm_train.shape)
print("LSTM validation:", X_lstm_val.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model_lstm = Sequential([
    Input(shape=(lookback, 1)),

    LSTM(64),

    Dropout(0.2),

    Dense(32, activation='relu'),

    Dense(1)
])

model_lstm.compile(
    optimizer='adam',
    loss='mse'
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model_lstm.fit(
    X_lstm_train,
    y_lstm_train,

    epochs=100,
    batch_size=32,

    validation_split=0.1,

    shuffle=False,

    callbacks=[early_stop],

    verbose=1
)

In [ ]:
pred_lstm_cal_scaled = model_lstm.predict(
    X_lstm_cal,
    verbose=0
).flatten()

pred_lstm_test_scaled = model_lstm.predict(
    X_lstm_test,
    verbose=0
).flatten()

pred_lstm_cal = scaler.inverse_transform(
    pred_lstm_cal_scaled.reshape(-1, 1)
).flatten()

pred_lstm_test = scaler.inverse_transform(
    pred_lstm_test_scaled.reshape(-1, 1)
).flatten()

In [ ]:
# ============================================
# ALIGN CALIBRATION PREDICTIONS
# ============================================

lgb_cal_indices = df.index[cal_mask]

lgb_cal_series = pd.Series(
    pred_lgb_cal,
    index=lgb_cal_indices
)

lstm_cal_series = pd.Series(
    pred_lstm_cal,
    index=idx_lstm_cal
)

common_cal = (
    lgb_cal_series.index
    .intersection(lstm_cal_series.index)
)

actual_cal = df.loc[
    common_cal,
    'revenue'
].values

lgb_cal = lgb_cal_series.loc[
    common_cal
].values

lstm_cal = lstm_cal_series.loc[
    common_cal
].values

print(len(common_cal))

In [ ]:
# ============================================
# ADAPTIVE FUSION
# ============================================

window = 8

adaptive_cal_pred = []
adaptive_weights_lgb = []
adaptive_weights_lstm = []

for i in range(len(actual_cal)):

    if i < window:

        w_lgb = 0.5
        w_lstm = 0.5

    else:

        lgb_error = np.sqrt(
            np.mean(
                (
                    actual_cal[i-window:i]
                    -
                    lgb_cal[i-window:i]
                ) ** 2
            )
        )

        lstm_error = np.sqrt(
            np.mean(
                (
                    actual_cal[i-window:i]
                    -
                    lstm_cal[i-window:i]
                ) ** 2
            )
        )

        # Inverse-error weighting
        inv_lgb = 1 / (lgb_error + 1e-8)
        inv_lstm = 1 / (lstm_error + 1e-8)

        total = inv_lgb + inv_lstm

        w_lgb = inv_lgb / total
        w_lstm = inv_lstm / total

    prediction = (
        w_lgb * lgb_cal[i]
        +
        w_lstm * lstm_cal[i]
    )

    adaptive_cal_pred.append(
        prediction
    )

    adaptive_weights_lgb.append(
        w_lgb
    )

    adaptive_weights_lstm.append(
        w_lstm
    )

adaptive_cal_pred = np.array(
    adaptive_cal_pred
)

adaptive_weights_lgb = np.array(
    adaptive_weights_lgb
)

adaptive_weights_lstm = np.array(
    adaptive_weights_lstm
)

print(
    "Average LightGBM weight:",
    adaptive_weights_lgb.mean()
)

print(
    "Average LSTM weight:",
    adaptive_weights_lstm.mean()
)

In [ ]:
best_weight = None
best_rmse = float("inf")

for w in np.arange(0, 1.01, 0.01):

    hybrid_pred = (
        w * pred_lgb_aligned
        + (1 - w) * pred_lstm_aligned
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            hybrid_pred
        )
    )

    if rmse < best_rmse:
        best_rmse = rmse
        best_weight = w

print("Best LightGBM weight:", best_weight)
print("Best LSTM weight:", 1 - best_weight)
print("Best RMSE:", best_rmse)

In [ ]:
hybrid_pred = (
    best_weight * pred_lgb_aligned
    + (1 - best_weight) * pred_lstm_aligned
)

In [ ]:
mae = mean_absolute_error(
    actual,
    hybrid_pred
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        hybrid_pred
    )
)

# Avoid division by zero
non_zero_mask = actual != 0

mape = np.mean(
    np.abs(
        (actual[non_zero_mask] - hybrid_pred[non_zero_mask])
        / actual[non_zero_mask]
    )
) * 100

print("Hybrid MAE :", mae)
print("Hybrid RMSE:", rmse)
print("Hybrid MAPE:", mape)

In [ ]:
# ============================================
# ALIGN TEST PREDICTIONS
# ============================================

lgb_test_indices = df.index[test_mask]

lgb_test_series = pd.Series(
    pred_lgb_test,
    index=lgb_test_indices
)

lstm_test_series = pd.Series(
    pred_lstm_test,
    index=idx_lstm_test
)

common_test = (
    lgb_test_series.index
    .intersection(lstm_test_series.index)
)

actual_test = df.loc[
    common_test,
    'revenue'
].values

lgb_test = lgb_test_series.loc[
    common_test
].values

lstm_test = lstm_test_series.loc[
    common_test
].values

In [ ]:
# ============================================
# ROLLING ADAPTIVE FUSION ON TEST
# ============================================

history_actual = list(actual_cal)
history_lgb = list(lgb_cal)
history_lstm = list(lstm_cal)

adaptive_test_pred = []
test_weights_lgb = []
test_weights_lstm = []

window = 8

for i in range(len(actual_test)):

    recent_actual = np.array(
        history_actual[-window:]
    )

    recent_lgb = np.array(
        history_lgb[-window:]
    )

    recent_lstm = np.array(
        history_lstm[-window:]
    )

    lgb_error = np.sqrt(
        np.mean(
            (recent_actual - recent_lgb) ** 2
        )
    )

    lstm_error = np.sqrt(
        np.mean(
            (recent_actual - recent_lstm) ** 2
        )
    )

    inv_lgb = 1 / (lgb_error + 1e-8)
    inv_lstm = 1 / (lstm_error + 1e-8)

    total = inv_lgb + inv_lstm

    w_lgb = inv_lgb / total
    w_lstm = inv_lstm / total

    prediction = (
        w_lgb * lgb_test[i]
        +
        w_lstm * lstm_test[i]
    )

    adaptive_test_pred.append(
        prediction
    )

    test_weights_lgb.append(
        w_lgb
    )

    test_weights_lstm.append(
        w_lstm
    )

    # After observing actual t,
    # it becomes available for t+1
    history_actual.append(
        actual_test[i]
    )

    history_lgb.append(
        lgb_test[i]
    )

    history_lstm.append(
        lstm_test[i]
    )

adaptive_test_pred = np.array(
    adaptive_test_pred
)

test_weights_lgb = np.array(
    test_weights_lgb
)

test_weights_lstm = np.array(
    test_weights_lstm
)

In [ ]:
# ============================================
# EVALUATION
# ============================================

def evaluate_model(
    actual,
    prediction,
    name
):

    mae = mean_absolute_error(
        actual,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            prediction
        )
    )

    nonzero = actual != 0

    mape = np.mean(
        np.abs(
            (
                actual[nonzero]
                -
                prediction[nonzero]
            )
            /
            actual[nonzero]
        )
    ) * 100

    return {
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape
    }


results = []

results.append(
    evaluate_model(
        actual_test,
        lgb_test,
        'LightGBM'
    )
)

results.append(
    evaluate_model(
        actual_test,
        lstm_test,
        'LSTM'
    )
)

# Static 50/50 baseline
static_pred = (
    0.5 * lgb_test
    +
    0.5 * lstm_test
)

results.append(
    evaluate_model(
        actual_test,
        static_pred,
        'Static Fusion'
    )
)

# Proposed method
results.append(
    evaluate_model(
        actual_test,
        adaptive_test_pred,
        'Adaptive Fusion'
    )
)

results_df = pd.DataFrame(
    results
)

results_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    test_weights_lgb,
    label='LightGBM Weight'
)

plt.plot(
    test_weights_lstm,
    label='LSTM Weight'
)

plt.xlabel('Test Time Step')
plt.ylabel('Weight')
plt.title(
    'Adaptive Fusion Weights Over Time'
)

plt.legend()
plt.grid(True)

plt.show()